## **CHALLENGES FACED BY ADOLESCENT BOYS AND GIRLS IN KENYA**

#### **DATA GENERATION PROCESS**

#### **INSTALL THE NEEDED LIBRARIES**

In [2]:
#!pip install pandas
#!pip install numpy

#### **LOAD THE NEEDED LIBRARIES**

In [3]:
import pandas as pd
import numpy as np

In [4]:
# ==========================================
# 1. CONFIGURATION & HELPER FUNCTIONS
# ==========================================
np.random.seed(42) # For reproducibility
N = 2500 # Target > 2200 observations

def generate_likert(latent_mean, noise_scale=1.0):
    """
    Converts a continuous latent mean into a 1-5 Likert scale.
    Adds normal noise, clips extremes, and rounds to nearest integer.
    """
    raw = latent_mean + np.random.normal(0, noise_scale, N)
    raw = np.clip(raw, -3.5, 3.5)
    # Shift to center around 3, then clip to 1-5
    likert = np.clip(np.round(raw + 3), 1, 5).astype(int)
    return likert

def weighted_choice_fast(options, weights_matrix):
    """
    Vectorized categorical sampling based on row-wise weights.
    Used to generate meaningful follow-up categorical responses.
    """
    weights_matrix = np.abs(weights_matrix) + 1e-5 # Ensure positive weights
    probs = weights_matrix / weights_matrix.sum(axis=1, keepdims=True)
    cumprobs = np.cumsum(probs, axis=1)
    rand = np.random.rand(len(probs), 1)
    indices = np.argmax(cumprobs > rand, axis=1)
    return np.array(options)[indices]

# ==========================================
# 2. DEMOGRAPHICS (SECTION A)
# ==========================================
age_groups = ['10-12 years', '13-15 years', '16-17 years', '18-19 years']
A1 = np.random.choice(age_groups, N, p=[0.15, 0.35, 0.35, 0.15])
A1_num = np.array([0, 1, 2, 3])[np.random.choice(4, N, p=[0.15, 0.35, 0.35, 0.15])]

A2 = np.random.choice(['Male', 'Female'], N, p=[0.48, 0.52])
A2_num = (A2 == 'Female').astype(int) # 0=Male, 1=Female

counties = ['Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Eldoret', 'Machakos', 'Nyeri', 'Garissa']
A3 = np.random.choice(counties, N)

A4 = np.random.choice(['Urban', 'Peri-urban', 'Rural'], N, p=[0.3, 0.3, 0.4])
A4_num = np.array([2, 1, 0])[np.searchsorted([0.3, 0.6], np.random.rand(N))] # 2=Urban, 1=Peri, 0=Rural

A5 = np.random.choice(['Both parents', 'Mother only', 'Father only', 'Other relatives', 'Guardian', 'Independent'], 
                      N, p=[0.4, 0.2, 0.1, 0.15, 0.1, 0.05])
A6 = np.random.choice(['Primary school', 'Secondary school', 'College/TVET', 'Not currently attending'], 
                      N, p=[0.2, 0.6, 0.1, 0.1])
A7 = np.random.choice(['Class 7', 'Class 8', 'Form 1', 'Form 2', 'Form 3', 'Form 4', 'TVET/College', 'None'], N)
A8 = np.random.choice(['1-3', '4-6', '7-9', '10 or more'], N, p=[0.2, 0.5, 0.2, 0.1])
A9 = np.random.choice(['Formal employment', 'Business/self-employment', 'Farming', 'Casual employment', 'Unemployed'], 
                      N, p=[0.2, 0.3, 0.2, 0.2, 0.1])
A10 = np.random.choice(['Very difficult', 'Difficult', 'Fair', 'Good', 'Very good'], N, p=[0.15, 0.25, 0.35, 0.2, 0.05])
A10_num = np.array([1, 2, 3, 4, 5])[np.searchsorted([0.15, 0.4, 0.75, 0.95], np.random.rand(N))]

# ==========================================
# 3. LATENT VARIABLES (The "Engine" of Correlations)
# ==========================================
# Base standard normal distributions
L_family_support = np.random.normal(0, 1, N)
L_family_conflict = np.random.normal(0, 1, N)
L_peer_pressure = np.random.normal(0, 1, N)
L_academic_pressure = np.random.normal(0, 1, N)
L_psych_distress = np.random.normal(0, 1, N)
L_risk_behavior = np.random.normal(0, 1, N)
L_health_barrier = np.random.normal(0, 1, N)
L_economic_stress = np.random.normal(0, 1, N)

# Inject structural correlations (e.g., distress is driven by conflict, peer pressure, and economics)
L_psych_distress += 0.4 * L_family_conflict + 0.3 * L_peer_pressure + 0.3 * L_academic_pressure + 0.4 * L_economic_stress
L_risk_behavior += 0.5 * L_peer_pressure + 0.3 * L_family_conflict
L_family_support -= 0.5 * L_family_conflict

# ==========================================
# 4. LIKERT ITEMS (SECTIONS B - J)
# ==========================================
# Section B: Educational
B1 = generate_likert(L_academic_pressure + 0.4 * A2_num) # Girls feel more pressure
B2 = generate_likert(0.3 * L_family_support + 0.4 * A10_num)
B3 = generate_likert(-0.4 * A10_num + 0.5 * L_economic_stress)
B4 = generate_likert(0.6 * L_psych_distress)
B5 = generate_likert(0.3 * L_family_support + 0.2 * A4_num)
B6 = generate_likert(0.4 * L_family_support)
B7 = generate_likert(0.4 * L_family_conflict + 0.2 * A1_num)
B8 = generate_likert(0.4 * L_psych_distress + 0.4 * L_academic_pressure)
B9 = generate_likert(0.5 * L_economic_stress + 0.3 * L_family_conflict)
B10 = generate_likert(0.4 * A10_num + 0.3 * L_family_support)

# Section C: Family
C1 = generate_likert(0.8 * L_family_support)
C2 = generate_likert(0.7 * L_family_support)
C3 = generate_likert(0.7 * L_family_support)
C4 = generate_likert(-0.6 * L_family_support + 0.8 * L_family_conflict)
C5 = generate_likert(0.6 * L_family_support - 0.5 * L_family_conflict)
C6 = generate_likert(0.5 * L_academic_pressure + 0.4 * A2_num) # Girls face higher expectations
C7 = generate_likert(0.7 * L_family_support)
C8 = generate_likert(0.6 * L_economic_stress)
C9 = generate_likert(0.5 * L_family_support - 0.2 * A1_num)
C10 = generate_likert(0.8 * L_family_support)

# Section D: Peer
D1 = generate_likert(0.4 * L_family_support)
D2 = generate_likert(0.8 * L_peer_pressure)
D3 = generate_likert(0.5 * L_psych_distress + 0.4 * L_peer_pressure)
D4 = generate_likert(0.5 * L_peer_pressure - 0.3 * A2_num) # Boys experience more teasing
D5 = generate_likert(0.6 * L_peer_pressure - 0.4 * A2_num) # Boys experience more bullying
D6 = generate_likert(0.5 * L_peer_pressure + 0.6 * A2_num) # Girls experience more social media pressure
D7 = generate_likert(-0.5 * L_psych_distress)
D8 = generate_likert(0.6 * L_peer_pressure + 0.4 * L_risk_behavior)
D9 = generate_likert(-0.5 * L_peer_pressure + 0.4 * L_family_support)
D10 = generate_likert(0.4 * L_family_support + 0.3 * A10_num)

# Section E: Health
E1 = generate_likert(0.3 * A10_num + 0.2 * A4_num)
E2 = generate_likert(0.3 * A10_num + 0.2 * A1_num)
E3 = generate_likert(0.4 * L_family_support + 0.3 * A10_num)
E4 = generate_likert(-0.5 * A10_num + 0.6 * L_health_barrier)
E5 = generate_likert(0.5 * A2_num + 0.5 * L_health_barrier) # Girls fear judgment more
E6 = generate_likert(0.3 * A10_num + 0.2 * A4_num)
E7 = generate_likert(0.3 * A10_num + 0.2 * A4_num)
E8 = generate_likert(0.4 * L_psych_distress + 0.4 * L_economic_stress)
E9 = generate_likert(-0.4 * A10_num + 0.5 * L_health_barrier)
E10 = generate_likert(0.4 * L_family_support + 0.3 * A10_num)

# Section F: Substance/Risk
F1 = generate_likert(0.5 * L_risk_behavior - 0.3 * A2_num + 0.2 * A4_num) # Boys, Urban
F2 = generate_likert(0.5 * L_peer_pressure + 0.5 * L_risk_behavior)
F3 = generate_likert(0.5 * L_risk_behavior + 0.2 * A4_num)
F4 = generate_likert(0.5 * L_peer_pressure)
F5 = generate_likert(0.5 * L_peer_pressure + 0.4 * L_risk_behavior)
F6 = generate_likert(0.5 * L_psych_distress + 0.4 * L_risk_behavior)
F7 = generate_likert(0.4 * A10_num)
F8 = generate_likert(0.4 * L_family_support + 0.3 * A10_num)
F9 = generate_likert(0.4 * L_health_barrier + 0.4 * L_risk_behavior)
F10 = generate_likert(0.4 * L_risk_behavior + 0.3 * L_psych_distress)

# Section G: Psychosocial
G1 = generate_likert(0.8 * L_psych_distress)
G2 = generate_likert(0.5 * L_psych_distress + 0.4 * L_economic_stress)
G3 = generate_likert(0.7 * L_psych_distress)
G4 = generate_likert(0.6 * L_psych_distress - 0.4 * L_family_support)
G5 = generate_likert(-0.6 * L_psych_distress + 0.4 * L_family_support)
G6 = generate_likert(0.7 * L_family_support)
G7 = generate_likert(0.4 * L_psych_distress - 0.4 * A2_num) # Boys hide emotions more
G8 = generate_likert(0.4 * L_family_support + 0.3 * A10_num)
G9 = generate_likert(0.5 * L_psych_distress - 0.3 * A2_num) # Boys hide problems more
G10 = generate_likert(-0.6 * L_psych_distress + 0.4 * L_family_support)

# Section H: Economic
H1 = generate_likert(0.8 * L_economic_stress)
H2 = generate_likert(0.6 * L_economic_stress - 0.4 * A10_num)
H3 = generate_likert(0.7 * L_economic_stress)
H4 = generate_likert(0.4 * L_economic_stress + 0.3 * A1_num)
H5 = generate_likert(0.4 * A10_num + 0.2 * A4_num)
H6 = generate_likert(0.4 * A10_num + 0.2 * A4_num)
H7 = generate_likert(0.7 * L_economic_stress)
H8 = generate_likert(0.5 * A10_num)
H9 = generate_likert(-0.5 * L_economic_stress + 0.4 * L_family_support)
H10 = generate_likert(0.6 * L_economic_stress)

# Section I: Support
I1 = generate_likert(0.5 * L_family_support + 0.3 * A10_num)
I2 = generate_likert(0.4 * A10_num + 0.2 * A4_num)
I3 = generate_likert(0.6 * L_family_support)
I4 = generate_likert(0.5 * A10_num)
I5 = generate_likert(-0.5 * L_health_barrier + 0.4 * L_family_support)
I6 = generate_likert(0.4 * A10_num)
I7 = generate_likert(0.7 * L_family_support)
I8 = generate_likert(0.4 * A10_num + 0.2 * A4_num)
I9 = generate_likert(0.3 * A10_num + 0.2 * A4_num)
I10 = generate_likert(0.5 * L_family_support - 0.3 * L_psych_distress)

# Section J: Priorities
J1 = generate_likert(0.4 * L_psych_distress + 0.4 * L_economic_stress)
J2 = generate_likert(0.5 * L_psych_distress)
J3 = generate_likert(0.5 * L_family_conflict)
J4 = generate_likert(0.5 * L_psych_distress + 0.4 * L_health_barrier)
J5 = generate_likert(0.6 * L_peer_pressure)
J6 = generate_likert(0.5 * L_health_barrier)
J7 = generate_likert(0.4 * L_economic_stress + 0.3 * A1_num)
J8 = generate_likert(0.4 * A1_num)
J9 = generate_likert(0.5 * L_risk_behavior)
J10 = generate_likert(0.3 * A1_num)

# ==========================================
# 5. CATEGORICAL FOLLOW-UPS
# ==========================================
# B Follow-ups
B11 = weighted_choice_fast(['Academic pressure', 'Lack of learning materials', 'Financial difficulties', 'Family responsibilities', 'Lack of teacher support', 'Concentration difficulties', 'Bullying'], 
                           np.column_stack([B1, 6-B2, B3, B7, 6-B6, B4, D5]))
B12 = weighted_choice_fast(['None', '1-2 days', '3-5 days', '6-10 days', 'More than 10 days'], 
                           np.column_stack([6-B9, np.ones(N), B9, B9**2, B9**3]))
B13 = weighted_choice_fast(['Parent/guardian', 'Teacher', 'School counsellor', 'Friend', 'Relative', 'Nobody'], 
                           np.column_stack([C1, B6, I2, D1, C10, G4]))

# C Follow-ups
C11 = weighted_choice_fast(['Mother', 'Father', 'Guardian', 'Sibling', 'Other relative', 'Nobody'], 
                           np.column_stack([np.ones(N), np.ones(N), np.ones(N), np.ones(N), np.ones(N), G4]))
C12 = weighted_choice_fast(['Never', 'Rarely', 'Sometimes', 'Often', 'Very often'], 
                           np.column_stack([6-C4, np.ones(N), C4, C4**2, C4**3]))
C13 = weighted_choice_fast(['Very uncomfortable', 'Uncomfortable', 'Neither', 'Comfortable', 'Very comfortable'], 
                           np.column_stack([C4, np.ones(N), np.ones(N), C1, C1**2]))

# D Follow-ups
D11 = weighted_choice_fast(['Peer pressure', 'Bullying', 'Social rejection', 'Teasing/insults', 'Pressure through social media', 'Friendship conflicts'], 
                           np.column_stack([D2, D5, D3, D4, D6, D8]))
D12 = weighted_choice_fast(['School', 'Home/neighbourhood', 'Online/social media', 'Community'], 
                           np.column_stack([B5, C4, D6, D1]))
D13 = weighted_choice_fast(['Parent/guardian', 'Teacher', 'Counsellor', 'Friend', 'Community member', 'Nobody'], 
                           np.column_stack([C1, B6, I2, D1, C10, G4]))

# E Follow-ups
E11 = weighted_choice_fast(['Parent/guardian', 'Teacher/school', 'Healthcare provider', 'Friends/peers', 'Internet/social media', 'Religious/community'], 
                           np.column_stack([C1, B6, E3, D1, D6, C10]))
E12 = weighted_choice_fast(['Cost', 'Distance', 'Lack of information', 'Fear of judgement', 'Lack of confidentiality', 'Family restrictions', 'Fear/embarrassment'], 
                           np.column_stack([E4, E4, E9, E5, 6-E7, C4, E5]))
E13 = weighted_choice_fast(['Very poor', 'Poor', 'Fair', 'Good', 'Very good'], 
                           np.column_stack([E9, E9, np.ones(N), E1, E1**2]))

# F Follow-ups
F11 = weighted_choice_fast(['Peer pressure', 'Family problems', 'Stress/emotional difficulties', 'Curiosity', 'Social media', 'Easy availability of substances', 'Economic difficulties'], 
                           np.column_stack([F2, C4, F6, np.ones(N), F4, F3, H1]))
F12 = weighted_choice_fast(['Parent/guardian', 'Teacher', 'Counsellor', 'Healthcare professional', 'Friend', 'Religious/community leader', 'Nobody'], 
                           np.column_stack([C1, B6, I2, E3, D1, C10, G4]))
F13 = weighted_choice_fast(['Very ineffective', 'Ineffective', 'Moderately effective', 'Effective', 'Very effective'], 
                           np.column_stack([F10, F10, np.ones(N), I8, I8**2]))

# G Follow-ups
G11 = weighted_choice_fast(['School/academic work', 'Family', 'Friendships/relationships', 'Financial difficulties', 'Future career', 'Social media', 'Health concerns'], 
                           np.column_stack([B1, C4, D3, H1, H4, D6, E8]))
G12 = weighted_choice_fast(['Talk to someone', 'Spend time alone', 'Pray/religious activity', 'Exercise/sports', 'Listen to music/watch entertainment', 'Use social media', 'Do nothing'], 
                           np.column_stack([G6, G4, C10, np.ones(N), np.ones(N), D6, G1]))
G13 = weighted_choice_fast(['Parent/guardian', 'Teacher/counsellor', 'Friend', 'Healthcare professional', 'Religious leader', 'Relative', 'Nobody'], 
                           np.column_stack([C1, I3, D1, E3, C10, C10, G4]))

# H Follow-ups
H11 = weighted_choice_fast(['Household poverty', 'Cost of education', 'Lack of employment opportunities', 'Lack of skills', 'Lack of financial support', 'Lack of career information'], 
                           np.column_stack([H1, H2, H4, 6-H6, H2, 6-H5]))
H12 = weighted_choice_fast(['Career guidance', 'Technical/vocational skills', 'Entrepreneurship training', 'Digital skills', 'Financial literacy', 'Mentorship', 'Academic support'], 
                           np.column_stack([H5, H6, H6, np.ones(N), np.ones(N), C10, B2]))
H13 = weighted_choice_fast(['Not at all confident', 'Slightly confident', 'Moderately confident', 'Very confident', 'Extremely confident'], 
                           np.column_stack([G2, G2, np.ones(N), H9, H9**2]))

# I Follow-ups
I11 = weighted_choice_fast(['Parent/guardian', 'Teacher/school counsellor', 'Friend', 'Healthcare provider', 'Religious leader', 'Community/youth organization', 'Online support', 'Nobody'], 
                           np.column_stack([C1, I3, D1, E3, C10, I9, D6, G4]))
I12 = weighted_choice_fast(['Fear of judgement', 'Lack of confidentiality', 'Cost', 'Lack of information', 'Distance', 'Family restrictions', 'Fear or embarrassment', 'Lack of services'], 
                           np.column_stack([I5, 6-I6, E4, 6-I1, E4, C4, I5, 6-I2]))
I13 = weighted_choice_fast(['Very difficult', 'Difficult', 'Neither difficult nor easy', 'Easy', 'Very easy'], 
                           np.column_stack([6-I1, 6-I1, np.ones(N), I8, I8**2]))

# J Follow-ups
J11 = weighted_choice_fast(['Lack of financial support', 'Peer pressure and bullying', 'Mental health issues', 'Academic pressure', 'Family conflicts', 'Substance abuse', 'Lack of career guidance'], 
                           np.column_stack([H1, D5, G1, B1, C4, F1, H5]))

# J12 requires picking 3 unique priorities
J12_opts = ['Education', 'Mental/emotional wellbeing', 'Physical health', 'Sexual and reproductive health', 'Substance-use prevention', 'Bullying and violence prevention', 'Family support', 'Employment/career preparation', 'Financial support', 'Digital/social-media challenges']
J12_w = np.column_stack([B3, G1, E8, E2, F1, D5, C1, H4, H1, D6])
probs_J12 = J12_w / J12_w.sum(axis=1, keepdims=True)
J12_mat = np.empty((N, 3), dtype=object)
for i in range(N):
    chosen = np.random.choice(len(J12_opts), 3, replace=False, p=probs_J12[i])
    J12_mat[i] = np.array(J12_opts)[chosen]
J12_1, J12_2, J12_3 = J12_mat[:, 0], J12_mat[:, 1], J12_mat[:, 2]

J13 = weighted_choice_fast(['Parents/guardians', 'Schools', 'Government', 'Healthcare providers', 'Community organizations', 'Religious organizations', 'Adolescents themselves', 'Shared responsibility among all stakeholders'], 
                           np.column_stack([C1, B5, np.ones(N), E3, I9, C10, J8, np.ones(N)]))

# ==========================================
# 6. OPEN-ENDED TEXT SIMULATION (Mixed-Methods Qualitative Arm)
# ==========================================
B14 = np.random.choice(['More financial aid for books', 'Better teacher support', 'Reduced academic pressure', 'More career guidance', 'Safe study environments'], N)
C14 = np.random.choice(['More family time', 'Better communication', 'Financial support', 'Understanding of youth issues', 'Less conflict'], N)
D14 = np.random.choice(['Anti-bullying programs', 'Peer counseling', 'Safe spaces', 'Parental education', 'Strict school policies'], N)
E14 = np.random.choice(['Free health services', 'Youth-friendly clinics', 'Better sex education', 'Confidential services', 'More health info'], N)
F14 = np.random.choice(['Rehab centers', 'Peer education', 'Sports and recreation', 'Mental health support', 'Strict drug policies'], N)
G14 = np.random.choice(['School counselors', 'Mental health apps', 'Peer support groups', 'Parental training', 'Safe spaces to talk'], N)
H14 = np.random.choice(['Scholarships', 'Vocational training', 'Mentorship programs', 'Entrepreneurship funds', 'Career fairs'], N)
I14 = np.random.choice(['More school counselors', 'Community youth centers', 'Online helplines', 'Parental awareness', 'Free services'], N)
J14 = np.random.choice(['Free education', 'Better mental health care', 'Job opportunities', 'Safe communities', 'Family support'], N)

# ==========================================
# 7. RESEARCHER SECTION (R1 - R6)
# ==========================================
R1 = np.arange(1, N + 1)
days_in_2026 = 277 # Up to Oct 4, 2026
R2_dates = pd.to_datetime('2026-01-01') + pd.to_timedelta(np.random.randint(0, days_in_2026, N), unit='D')
R2 = R2_dates.strftime('%d/%m/%Y')
R3 = A3 # Location matches County
R4 = np.random.choice(['Self-administered', 'Interviewer-administered', 'Electronic questionnaire'], N, p=[0.3, 0.4, 0.3])
R5 = np.random.choice(['Yes', 'Partially', 'No'], N, p=[0.92, 0.06, 0.02])
R6 = np.random.choice(['None', 'Respondent seemed distracted', 'Environment was noisy', 'Respondent required clarification', 'Completed without issues'], N, p=[0.6, 0.1, 0.1, 0.1, 0.1])

# ==========================================
# 8. ASSEMBLE DATAFRAME & EXPORT
# ==========================================
df = pd.DataFrame({
    # Section A
    'A1_Age': A1, 'A2_Sex': A2, 'A3_County': A3, 'A4_Residence': A4,
    'A5_Living_arrangement': A5, 'A6_Education_status': A6, 'A7_Class_Level': A7,
    'A8_Household_size': A8, 'A9_Parent_occupation': A9, 'A10_Economic_situation': A10,
    
    # Section B
    'B1': B1, 'B2': B2, 'B3': B3, 'B4': B4, 'B5': B5, 'B6': B6, 'B7': B7, 'B8': B8, 'B9': B9, 'B10': B10,
    'B11_Main_challenge': B11, 'B12_Days_missed': B12, 'B13_Approach_who': B13, 'B14_Improvement': B14,
    
    # Section C
    'C1': C1, 'C2': C2, 'C3': C3, 'C4': C4, 'C5': C5, 'C6': C6, 'C7': C7, 'C8': C8, 'C9': C9, 'C10': C10,
    'C11_Support_who': C11, 'C12_Conflict_freq': C12, 'C13_Comfort_discuss': C13, 'C14_Support_needed': C14,
    
    # Section D
    'D1': D1, 'D2': D2, 'D3': D3, 'D4': D4, 'D5': D5, 'D6': D6, 'D7': D7, 'D8': D8, 'D9': D9, 'D10': D10,
    'D11_Peer_challenge': D11, 'D12_Location': D12, 'D13_Tell_who': D13, 'D14_What_would_help': D14,
    
    # Section E
    'E1': E1, 'E2': E2, 'E3': E3, 'E4': E4, 'E5': E5, 'E6': E6, 'E7': E7, 'E8': E8, 'E9': E9, 'E10': E10,
    'E11_Info_source': E11, 'E12_Main_barrier': E12, 'E13_Availability': E13, 'E14_Improvement': E14,
    
    # Section F
    'F1': F1, 'F2': F2, 'F3': F3, 'F4': F4, 'F5': F5, 'F6': F6, 'F7': F7, 'F8': F8, 'F9': F9, 'F10': F10,
    'F11_Contributing_factor': F11, 'F12_Seek_help_who': F12, 'F13_Program_effectiveness': F13, 'F14_Intervention': F14,
    
    # Section G
    'G1': G1, 'G2': G2, 'G3': G3, 'G4': G4, 'G5': G5, 'G6': G6, 'G7': G7, 'G8': G8, 'G9': G9, 'G10': G10,
    'G11_Source_pressure': G11, 'G12_Coping_mechanism': G12, 'G13_Talk_to_who': G13, 'G14_Support_type': G14,
    
    # Section H
    'H1': H1, 'H2': H2, 'H3': H3, 'H4': H4, 'H5': H5, 'H6': H6, 'H7': H7, 'H8': H8, 'H9': H9, 'H10': H10,
    'H11_Econ_challenge': H11, 'H12_Preparation_area': H12, 'H13_Confidence_future': H13, 'H14_Opportunity': H14,
    
    # Section I
    'I1': I1, 'I2': I2, 'I3': I3, 'I4': I4, 'I5': I5, 'I6': I6, 'I7': I7, 'I8': I8, 'I9': I9, 'I10': I10,
    'I11_Support_source': I11, 'I12_Barrier_help': I12, 'I13_Ease_find_info': I13, 'I14_Accessibility': I14,
    
    # Section J
    'J1': J1, 'J2': J2, 'J3': J3, 'J4': J4, 'J5': J5, 'J6': J6, 'J7': J7, 'J8': J8, 'J9': J9, 'J10': J10,
    'J11_Biggest_challenge': J11, 'J12_Priority_1': J12_1, 'J12_Priority_2': J12_2, 'J12_Priority_3': J12_3,
    'J13_Responsibility': J13, 'J14_Important_change': J14,
    
    # Researcher Section
    'R1_ID': R1, 'R2_Date': R2, 'R3_Location': R3, 'R4_Mode': R4, 'R5_Completed': R5, 'R6_Observations': R6
})

In [5]:
df.head(20)

,A1_Age,A2_Sex,A3_County,A4_Residence,A5_Living_arrangement,A6_Education_status,A7_Class_Level,A8_Household_size,A9_Parent_occupation,A10_Economic_situation,...,J12_Priority_2,J12_Priority_3,J13_Responsibility,J14_Important_change,R1_ID,R2_Date,R3_Location,R4_Mode,R5_Completed,R6_Observations
0,13-15 years,Male,Nakuru,Rural,Both parents,Secondary school,None,7-9,Business/self-employment,Very difficult,...,Mental/emotional wellbeing,Employment/career preparation,Adolescents themselves,Better mental health care,1,19/06/2026,Nakuru,Self-administered,Yes,None
1,18-19 years,Male,Nakuru,Urban,Both parents,Secondary school,Form 2,1-3,Formal employment,Very difficult,...,Family support,Education,Religious organizations,Safe communities,2,29/06/2026,Nakuru,Interviewer-administered,Yes,Completed without issues
2,16-17 years,Female,Mombasa,Urban,Both parents,Primary school,Class 7,4-6,Business/self-employment,Fair,...,Sexual and reproductive health,Mental/emotional wellbeing,Adolescents themselves,Job opportunities,3,01/10/2026,Mombasa,Electronic questionnaire,Yes,Respondent required clarification
3,16-17 years,Male,Eldoret,Urban,Both parents,Secondary school,Class 8,4-6,Casual employment,Very good,...,Employment/career preparation,Financial support,Religious organizations,Family support,4,16/09/2026,Eldoret,Electronic questionnaire,Yes,None
4,13-15 years,Female,Nairobi,Rural,Both parents,Primary school,Form 1,4-6,Business/self-employment,Fair,...,Sexual and reproductive health,Substance-use prevention,Community organizations,Safe communities,5,02/01/2026,Nairobi,Electronic questionnaire,Partially,Environment was noisy
5,13-15 years,Male,Nairobi,Rural,Mother only,Not currently attending,Form 4,7-9,Farming,Very difficult,...,Family support,Substance-use prevention,Schools,Job opportunities,6,26/06/2026,Nairobi,Electronic questionnaire,Yes,Respondent seemed distracted
6,10-12 years,Female,Eldoret,Urban,Both parents,Secondary school,Form 2,10 or more,Farming,Difficult,...,Physical health,Mental/emotional wellbeing,Healthcare providers,Job opportunities,7,28/09/2026,Eldoret,Interviewer-administered,Yes,None
7,18-19 years,Female,Machakos,Peri-urban,Other relatives,Primary school,Form 2,1-3,Casual employment,Difficult,...,Financial support,Digital/social-media challenges,Schools,Free education,8,05/07/2026,Machakos,Electronic questionnaire,Yes,None
8,16-17 years,Male,Mombasa,Urban,Both parents,Primary school,TVET/College,7-9,Formal employment,Difficult,...,Physical health,Education,Adolescents themselves,Safe communities,9,15/04/2026,Mombasa,Interviewer-administered,Yes,None
9,16-17 years,Male,Mombasa,Peri-urban,Mother only,Secondary school,Class 7,4-6,Business/self-employment,Very difficult,...,Sexual and reproductive health,Substance-use prevention,Parents/guardians,Family support,10,25/03/2026,Mombasa,Electronic questionnaire,Yes,None


In [ ]:
print(f"✅ Dataset generated successfully!")
print(f"📊 Observations: {len(df)}")
print(f"📝 Variables: {len(df.columns)}")
print(f"💾 Saved to: {filename}")

In [6]:
# Save to CSV
filename = 'Mediacrest_Adolescent_Challenges_Data.csv'
df.to_csv(filename, index=False)